In [4]:
"""
High-accuracy ensemble pipeline for predicting academic success.
Target: surpass 0.83758 accuracy on the test set.
Strategy: rich feature engineering + 5-fold stacking with XGB, LightGBM, CatBoost, HGB, ET
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
)

RANDOM_STATE = 42
N_SPLITS = 5

# ── Load data ──────────────────────────────────────────────────────────────────
train_df = pd.read_csv("train_assignment.csv")
test_df = pd.read_csv("test_assignment.csv")

# ── Feature engineering ────────────────────────────────────────────────────────
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Semester totals
    df["cu_total_enrolled"] = df["cu1_enrolled"] + df["cu2_enrolled"]
    df["cu_total_evaluations"] = df["cu1_evaluations"] + df["cu2_evaluations"]
    df["cu_total_approved"] = df["cu1_approved"] + df["cu2_approved"]
    df["cu_total_credited"] = df["cu1_credited"] + df["cu2_credited"]
    df["cu_total_without_eval"] = df["cu1_without_evaluations"] + df["cu2_without_evaluations"]

    # Smoothed approval rates (Laplace smoothing)
    eps = 1.0
    df["cu1_approval_rate_enr"] = (df["cu1_approved"] + eps) / (df["cu1_enrolled"] + 2 * eps)
    df["cu2_approval_rate_enr"] = (df["cu2_approved"] + eps) / (df["cu2_enrolled"] + 2 * eps)
    df["cu1_approval_rate_eval"] = (df["cu1_approved"] + eps) / (df["cu1_evaluations"] + 2 * eps)
    df["cu2_approval_rate_eval"] = (df["cu2_approved"] + eps) / (df["cu2_evaluations"] + 2 * eps)
    df["cu_total_approval_rate_enr"] = (df["cu_total_approved"] + eps) / (df["cu_total_enrolled"] + 2 * eps)
    df["cu_total_approval_rate_eval"] = (df["cu_total_approved"] + eps) / (df["cu_total_evaluations"] + 2 * eps)

    # Grade features
    df["cu_grade_mean"] = (df["cu1_grade"] + df["cu2_grade"]) / 2.0
    df["cu_grade_gap"] = df["cu1_grade"] - df["cu2_grade"]
    df["cu_grade_max"] = np.maximum(df["cu1_grade"], df["cu2_grade"])
    df["cu_grade_min"] = np.minimum(df["cu1_grade"], df["cu2_grade"])

    # Grade per approved unit (how well they did per pass)
    df["cu1_grade_per_approved"] = df["cu1_grade"] / (df["cu1_approved"] + 1)
    df["cu2_grade_per_approved"] = df["cu2_grade"] / (df["cu2_approved"] + 1)

    # Semester difference features (progress sem1 -> sem2)
    df["cu_approved_diff"] = df["cu2_approved"] - df["cu1_approved"]
    df["cu_enrolled_diff"] = df["cu2_enrolled"] - df["cu1_enrolled"]
    df["cu_grade_diff"] = df["cu2_grade"] - df["cu1_grade"]
    df["cu_eval_diff"] = df["cu2_evaluations"] - df["cu1_evaluations"]

    # Failure / dropout risk signals
    df["cu1_fail_count"] = df["cu1_enrolled"] - df["cu1_approved"]
    df["cu2_fail_count"] = df["cu2_enrolled"] - df["cu2_approved"]
    df["cu_total_fail_count"] = df["cu1_fail_count"] + df["cu2_fail_count"]
    df["cu_fail_rate"] = df["cu_total_fail_count"] / (df["cu_total_enrolled"] + 1)

    # Evaluation intensity
    df["cu1_eval_per_enrolled"] = df["cu1_evaluations"] / (df["cu1_enrolled"] + 1)
    df["cu2_eval_per_enrolled"] = df["cu2_evaluations"] / (df["cu2_enrolled"] + 1)

    # Financial risk
    df["financial_risk"] = df["debtor_flag"] * (1 - df["tuition_up_to_date_flag"])
    df["financial_support"] = df["scholarship_holder_flag"] * df["tuition_up_to_date_flag"]

    # Age interactions
    df["approved_per_age"] = df["cu_total_approved"] / (df["age_at_enrollment"] + 1)
    df["grade_per_age"] = df["cu_grade_mean"] / (df["age_at_enrollment"] + 1)

    # Engagement
    df["engagement_gap"] = df["cu_total_evaluations"] - df["cu_total_approved"]

    # Admission strength
    df["admission_x_approval"] = df["admission_grade"] * df["cu_total_approval_rate_enr"]
    df["prev_qual_x_approval"] = df["previous_qualification_grade"] * df["cu_total_approval_rate_enr"]
    df["grade_x_admission"] = df["cu_grade_mean"] * df["admission_grade"]

    # Binary: did the student improve from sem1 to sem2?
    df["improved_sem2"] = (df["cu2_approved"] > df["cu1_approved"]).astype(int)
    df["grade_improved_sem2"] = (df["cu2_grade"] > df["cu1_grade"]).astype(int)

    # Zero activity flags
    df["zero_approved_sem1"] = (df["cu1_approved"] == 0).astype(int)
    df["zero_approved_sem2"] = (df["cu2_approved"] == 0).astype(int)
    df["zero_approved_both"] = ((df["cu1_approved"] == 0) & (df["cu2_approved"] == 0)).astype(int)
    df["zero_grade_both"] = ((df["cu1_grade"] == 0) & (df["cu2_grade"] == 0)).astype(int)

    # Macroeconomic interactions
    df["gdp_x_unemployment"] = df["gdp"] * df["unemployment_rate"]
    df["inflation_x_unemployment"] = df["inflation_rate"] * df["unemployment_rate"]

    return df


train_fe = add_features(train_df)
test_fe = add_features(test_df)

target_col = "Target"
id_col = "id"
feature_cols = [c for c in train_fe.columns if c not in [target_col, id_col]]

X = train_fe[feature_cols].values
X_test = test_fe[feature_cols].values

le = LabelEncoder()
y = le.fit_transform(train_fe[target_col])
class_names = le.classes_
n_classes = len(class_names)

print(f"Features: {len(feature_cols)}, Train: {len(X)}, Test: {len(X_test)}, Classes: {list(class_names)}")

# ── Model definitions ─────────────────────────────────────────────────────────
def get_models():
    return {
        "xgb1": XGBClassifier(
            objective="multi:softprob",
            n_estimators=4000,
            learning_rate=0.02,
            max_depth=6,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=3.0,
            reg_alpha=0.3,
            gamma=0.3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric="mlogloss",
            tree_method="hist",
            early_stopping_rounds=100,
        ),
        "xgb2": XGBClassifier(
            objective="multi:softprob",
            n_estimators=3000,
            learning_rate=0.03,
            max_depth=7,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.75,
            reg_lambda=2.0,
            reg_alpha=0.5,
            gamma=0.2,
            random_state=RANDOM_STATE + 1,
            n_jobs=-1,
            eval_metric="mlogloss",
            tree_method="hist",
            early_stopping_rounds=80,
        ),
        "lgbm1": LGBMClassifier(
            n_estimators=4000,
            learning_rate=0.02,
            max_depth=7,
            num_leaves=63,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=3.0,
            reg_alpha=0.3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),
        "lgbm2": LGBMClassifier(
            n_estimators=3000,
            learning_rate=0.03,
            max_depth=6,
            num_leaves=50,
            min_child_samples=50,
            subsample=0.85,
            colsample_bytree=0.75,
            reg_lambda=2.0,
            reg_alpha=0.5,
            random_state=RANDOM_STATE + 2,
            n_jobs=-1,
            verbose=-1,
        ),
        "cat1": CatBoostClassifier(
            iterations=4000,
            learning_rate=0.03,
            depth=7,
            l2_leaf_reg=5.0,
            random_seed=RANDOM_STATE,
            verbose=0,
            auto_class_weights="Balanced",
            early_stopping_rounds=100,
        ),
        "cat2": CatBoostClassifier(
            iterations=3000,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            random_seed=RANDOM_STATE + 3,
            verbose=0,
            auto_class_weights="Balanced",
            early_stopping_rounds=80,
        ),
        "hgb": HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_depth=7,
            max_iter=2000,
            min_samples_leaf=30,
            l2_regularization=2.0,
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=50,
        ),
        "et": ExtraTreesClassifier(
            n_estimators=800,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_features=0.6,
            min_samples_leaf=2,
            class_weight="balanced",
        ),
    }


# ── K-fold stacking ───────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
model_names = list(get_models().keys())

oof_probas = {name: np.zeros((len(X), n_classes), dtype=np.float64) for name in model_names}
test_fold_probas = {name: [] for name in model_names}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold}/{N_SPLITS}  (train={len(tr_idx)}, val={len(va_idx)})")
    print(f"{'='*60}")

    fold_models = get_models()
    for name, model in fold_models.items():
        print(f"  Training {name}...", end=" ", flush=True)

        if "xgb" in name:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        elif "lgbm" in name:
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[
                    __import__("lightgbm").early_stopping(100, verbose=False),
                    __import__("lightgbm").log_evaluation(0),
                ],
            )
        elif "cat" in name:
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
        else:
            model.fit(X_tr, y_tr)

        va_prob = model.predict_proba(X_va)
        te_prob = model.predict_proba(X_test)

        oof_probas[name][va_idx] = va_prob
        test_fold_probas[name].append(te_prob)

        va_pred = np.argmax(va_prob, axis=1)
        acc = accuracy_score(y_va, va_pred)
        f1 = f1_score(y_va, va_pred, average="macro")
        print(f"acc={acc:.5f}  f1={f1:.5f}")

# ── OOF leaderboard ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("OOF Leaderboard")
print("=" * 60)

rows = []
test_probas_mean = {}
for name in model_names:
    oof_pred = np.argmax(oof_probas[name], axis=1)
    macro_f1 = f1_score(y, oof_pred, average="macro")
    acc = accuracy_score(y, oof_pred)
    rows.append((name, macro_f1, acc))
    test_probas_mean[name] = np.mean(np.stack(test_fold_probas[name], axis=0), axis=0)

score_df = pd.DataFrame(rows, columns=["model", "macro_f1", "accuracy"]).sort_values(
    by="accuracy", ascending=False
).reset_index(drop=True)

print(score_df.to_string(index=False))

# ── Ensemble A: Weighted soft vote (all models) ───────────────────────────────
# Weight by accuracy^8 to strongly favor best models
w = score_df.set_index("model")["accuracy"].to_dict()
weights = {name: w[name] ** 8 for name in model_names}
w_sum = sum(weights.values())
weights = {name: v / w_sum for name, v in weights.items()}

oof_soft = np.zeros((len(X), n_classes), dtype=np.float64)
test_soft = np.zeros((len(X_test), n_classes), dtype=np.float64)
for name in model_names:
    oof_soft += weights[name] * oof_probas[name]
    test_soft += weights[name] * test_probas_mean[name]

pred_soft = np.argmax(oof_soft, axis=1)
acc_soft = accuracy_score(y, pred_soft)
f1_soft = f1_score(y, pred_soft, average="macro")
print(f"\nSoft Vote Ensemble -> Accuracy: {acc_soft:.5f}, Macro-F1: {f1_soft:.5f}")

# ── Ensemble B: Stacking with LogisticRegression meta-learner ─────────────────
stack_X = np.hstack([oof_probas[name] for name in model_names])
stack_X_test = np.hstack([test_probas_mean[name] for name in model_names])

# Try multiple C values and pick best on OOF
best_stack_acc = 0
best_meta = None
for C_val in [0.1, 0.3, 0.5, 0.7, 1.0, 2.0, 5.0]:
    meta = LogisticRegression(
        C=C_val,
        max_iter=5000,
        multi_class="multinomial",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        solver="lbfgs",
    )
    meta.fit(stack_X, y)
    pred_meta = meta.predict(stack_X)
    acc_meta = accuracy_score(y, pred_meta)
    if acc_meta > best_stack_acc:
        best_stack_acc = acc_meta
        best_meta = meta
        best_C = C_val

pred_stack = best_meta.predict(stack_X)
acc_stack = accuracy_score(y, pred_stack)
f1_stack = f1_score(y, pred_stack, average="macro")
print(f"Stacking Ensemble (C={best_C}) -> Accuracy: {acc_stack:.5f}, Macro-F1: {f1_stack:.5f}")

# ── Ensemble C: Rank-averaged soft vote + stacking blend ─────────────────────
# Blend soft vote and stacking probabilities
alpha = 0.5  # try equal blend
test_stack_prob = best_meta.predict_proba(stack_X_test)
oof_stack_prob = best_meta.predict_proba(stack_X)

oof_blend = alpha * oof_soft + (1 - alpha) * oof_stack_prob
test_blend = alpha * test_soft + (1 - alpha) * test_stack_prob

pred_blend = np.argmax(oof_blend, axis=1)
acc_blend = accuracy_score(y, pred_blend)
f1_blend = f1_score(y, pred_blend, average="macro")
print(f"Blended (soft+stack) -> Accuracy: {acc_blend:.5f}, Macro-F1: {f1_blend:.5f}")

# Try different alpha values
best_alpha = alpha
best_blend_acc = acc_blend
for a in [0.3, 0.4, 0.5, 0.6, 0.7]:
    oof_b = a * oof_soft + (1 - a) * oof_stack_prob
    pred_b = np.argmax(oof_b, axis=1)
    acc_b = accuracy_score(y, pred_b)
    if acc_b > best_blend_acc:
        best_blend_acc = acc_b
        best_alpha = a

print(f"Best blend alpha={best_alpha}, accuracy={best_blend_acc:.5f}")

# ── Pick best ensemble strategy ──────────────────────────────────────────────
results = {
    "soft_vote": (acc_soft, test_soft),
    "stacking": (acc_stack, best_meta.predict_proba(stack_X_test)),
    "blend": (best_blend_acc, best_alpha * test_soft + (1 - best_alpha) * test_stack_prob),
}

best_name = max(results, key=lambda k: results[k][0])
best_acc = results[best_name][0]
best_test_prob = results[best_name][1]

print(f"\n*** Best ensemble: {best_name} with OOF accuracy {best_acc:.5f} ***")

# ── Generate submission ──────────────────────────────────────────────────────
test_pred_idx = np.argmax(best_test_prob, axis=1)
test_pred = le.inverse_transform(test_pred_idx)

submission = pd.DataFrame({
    "id": test_df["id"],
    "Target": test_pred,
})
submission.to_csv("submission.csv", index=False)

print(f"\nSaved submission.csv ({len(submission)} rows)")
print("Target distribution:")
print(submission["Target"].value_counts().to_string())

# Final OOF report
best_oof_pred = np.argmax(
    results[best_name][1] if best_name == "stacking"
    else (best_alpha * oof_soft + (1 - best_alpha) * oof_stack_prob if best_name == "blend" else oof_soft),
    axis=1
) if False else np.argmax(
    oof_soft if best_name == "soft_vote"
    else (best_meta.predict_proba(stack_X) if best_name == "stacking"
          else best_alpha * oof_soft + (1 - best_alpha) * oof_stack_prob),
    axis=1
)
print(f"\nOOF Classification Report ({best_name}):")
print(classification_report(y, best_oof_pred, target_names=class_names))


Features: 79, Train: 76518, Test: 51012, Classes: ['Dropout', 'Enrolled', 'Graduate']

Fold 1/5  (train=61214, val=15304)
  Training xgb1... acc=0.83011  f1=0.79187
  Training xgb2... acc=0.82946  f1=0.79095
  Training lgbm1... 

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


acc=0.83031  f1=0.79221
  Training lgbm2... 

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


acc=0.82835  f1=0.78991
  Training cat1... acc=0.81841  f1=0.79263
  Training cat2... acc=0.81737  f1=0.79207
  Training hgb... acc=0.82828  f1=0.78967
  Training et... 

KeyboardInterrupt: 